# Lab 01 — Next-token prediction and a tiny language model\n\n**Free-first:** designed for Google Colab or CPU.\n\nWe will not start with a giant Transformer. We will first build the smallest useful language model so that the training objective becomes concrete.\n\nLearning loop: **Predict → Run → Observe → Explain → Break → Decide**.\n

## 1. Imports\nWe use only PyTorch and standard Python. This is intentional: the first lab should expose the learning mechanics rather than hide them behind a trainer library.\n

In [ ]:
import math\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\ntorch.manual_seed(42)\ndevice = 'cuda' if torch.cuda.is_available() else 'cpu'\nprint('device:', device)\n

## 2. Our tiny corpus\nA language model needs sequences. For the first experiment, use a tiny public-domain corpus so the mechanics are visible.\n

In [ ]:
text = '''To be, or not to be, that is the question.\nWhether 'tis nobler in the mind to suffer\nThe slings and arrows of outrageous fortune,\nOr to take arms against a sea of troubles.\nTo be, or not to be.\n'''.lower()\nchars = sorted(set(text))\nvocab_size = len(chars)\nstoi = {ch:i for i,ch in enumerate(chars)}\nitos = {i:ch for ch,i in stoi.items()}\nencode = lambda s: [stoi[c] for c in s]\ndecode = lambda ids: ''.join(itos[i] for i in ids)\ndata = torch.tensor(encode(text), dtype=torch.long)\nprint('vocab size:', vocab_size)\nprint('tokens:', len(data))\n

## 3. Turn one sequence into supervised next-token examples\n\nFor a sequence like `abcd`, the training pairs are:\n\n`a → b`, `b → c`, `c → d`.\n\nFor language-model training, this pattern is applied across many positions and batches.\n

In [ ]:
block_size = 16\nbatch_size = 32\n\ndef get_batch(batch_size=batch_size, block_size=block_size):\n    ix = torch.randint(len(data) - block_size, (batch_size,))\n    x = torch.stack([data[i:i+block_size] for i in ix])\n    y = torch.stack([data[i+1:i+block_size+1] for i in ix])\n    return x.to(device), y.to(device)\n\nxb, yb = get_batch()\nprint('x shape:', xb.shape)\nprint('y shape:', yb.shape)\nprint('first x:', decode(xb[0].tolist()))\nprint('first y:', decode(yb[0].tolist()))\n

### Checkpoint question\n**Why is `y` shifted by one position?**\n\n**Answer:** because the supervised target at position `t` is the token that follows the input token/context at position `t`.\n

## 4. The smallest baseline: a bigram model\n\nBefore a Transformer, use a lookup table. The model only asks:\n\n> Given the current character/token, what token tends to come next?\n\nThis model has no attention and no hidden representation. That is exactly why it is useful.\n

In [ ]:
class BigramLanguageModel(nn.Module):\n    def __init__(self, vocab_size):\n        super().__init__()\n        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)\n\n    def forward(self, idx, targets=None):\n        logits = self.token_embedding_table(idx)\n        loss = None\n        if targets is not None:\n            B, T, C = logits.shape\n            logits = logits.view(B*T, C)\n            targets = targets.view(B*T)\n            loss = F.cross_entropy(logits, targets)\n        return logits, loss\n\nmodel = BigramLanguageModel(vocab_size).to(device)\nlogits, loss = model(xb, yb)\nprint('logits:', logits.shape)\nprint('initial loss:', float(loss))\nprint('theoretical random baseline ≈ log(V):', math.log(vocab_size))\n

## 5. Train it\n\nNotice what is happening: we are not telling the model rules. We are repeatedly comparing its probability distribution with the observed next token and updating the embedding table.\n

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)\n\nloss_history = []\nfor step in range(1000):\n    xb, yb = get_batch()\n    logits, loss = model(xb, yb)\n    optimizer.zero_grad(set_to_none=True)\n    loss.backward()\n    optimizer.step()\n    loss_history.append(float(loss))\n    if step % 200 == 0:\n        print(f'step {step:4d} | loss {loss.item():.4f}')\n

In [ ]:
import matplotlib.pyplot as plt\nplt.plot(loss_history)\nplt.xlabel('training step')\nplt.ylabel('cross-entropy loss')\nplt.title('Bigram training loss')\nplt.show()\n

## 6. Generate text\n\nThe generation loop is deliberately tiny. Predict one token, append it, predict again.\n

In [ ]:
@torch.no_grad()\ndef generate(model, idx, max_new_tokens):\n    for _ in range(max_new_tokens):\n        logits, _ = model(idx[:, -block_size:])\n        logits = logits[:, -1, :]\n        probs = F.softmax(logits, dim=-1)\n        idx_next = torch.multinomial(probs, num_samples=1)\n        idx = torch.cat((idx, idx_next), dim=1)\n    return idx\n\nstart = torch.tensor([[stoi['t']]], dtype=torch.long, device=device)\nsample = generate(model, start, 300)[0].tolist()\nprint(decode(sample))\n

## 7. Break it on purpose\n\nChange **one** thing at a time and predict what should happen before rerunning:\n\n1. Set learning rate to `1.0`.\n2. Train for only 10 steps.\n3. Train on a corpus containing only one repeated sentence.\n4. Reduce the corpus to a few characters.\n\nWrite your prediction first. Then run the experiment.\n

## 8. Critical-thinking answers\n\n**Q: Why can the model generate text it never saw verbatim?**\n\n**A:** It learns a conditional distribution rather than a simple lookup of complete sentences. Even a tiny bigram model combines learned local transition probabilities in new sequences. A Transformer can represent much richer conditional structure.\n\n**Q: Why is this not yet an LLM?**\n\n**A:** The model is far too small and the corpus far too limited. The point is to expose the training objective, not to pretend that scale is irrelevant.\n\n**Q: What is the biggest limitation of the bigram model?**\n\n**A:** It conditions only on the current token. It cannot directly model long-range dependencies such as agreement or relationships between distant words.\n

## 9. The bridge to Transformers\n\nWe now have the essential training loop:\n\n`tokens → logits → cross-entropy → gradients → weight update`.\n\nThe next lectures replace the simplistic representation and context mechanism with embeddings, positional information, self-attention, multi-head attention, MLPs, residual connections, normalization, and a full Transformer block.\n\n### Exit challenge\nExplain the entire lab to another student without saying the words **Transformer**, **GPT**, or **attention**. If you can do that, you understand the core objective rather than the brand names.\n